In [1]:
import numpy as np

from lib.read_data import cycles, u_desvio, y, y_desvio, y_ref

model_name = "ARX"


In [2]:
import hickle as hkl


def gerar_ARX(y, u):
    # Construindo a matriz de regressão
    X = np.hstack([y[:-1, :], u[:-1, :]])

    # Saídas futuras
    Y = y[1:, :]

    # Estimando parâmetros via mínimos quadrados
    theta = np.linalg.lstsq(X, Y)[0]
    return theta


def inferir_ARX(theta, y, u):
    N = len(y)
    y_arx = np.zeros_like(y)
    y_arx[0] = y[0]  # inicializa com o primeiro valor real

    for k in range(1, N):
        phi = np.concatenate([y_arx[k - 1], u[k - 1]])
        y_arx[k] = phi @ theta
    return y_arx


slice = int(len(y_desvio) * 0.5)
theta = gerar_ARX(y_desvio[:slice], u_desvio[:slice])
hkl.dump(theta, f"../export/{model_name}.hkl")

print("Parâmetros estimados:\n", theta)
# from lib.read_data import u_ref
# u_sp = np.array([610, 220, 135, 105]) - u_ref
# u_desvio = np.full((200, 4), u_sp)

y_arx = inferir_ARX(theta, y_desvio, u_desvio)

# Convertendo o valor desvio em variavel de engenharia
y_arx = y_arx + np.tile(y_ref, (y_arx.shape[0], 1))


Parâmetros estimados:
 [[ 0.36680423  0.79812824  2.5342445 ]
 [ 0.00291944  0.52521079  1.50011943]
 [ 0.01495648  0.0058226  -0.08998492]
 [-0.0073093   0.01769563 -0.03392627]
 [-0.01390491  0.05777156 -0.11353344]
 [ 0.00368511  0.01447885  0.04679554]
 [ 0.018839   -0.08710176  0.27048997]]


In [3]:
ny = y_desvio.shape[1]
nu = u_desvio.shape[1]

A = theta[:ny, :].T
B = theta[ny : ny + nu, :].T

print("Matriz A:")
print(A)

print("\nMatriz B:")
print(B)


Matriz A:
[[ 0.36680423  0.00291944  0.01495648]
 [ 0.79812824  0.52521079  0.0058226 ]
 [ 2.5342445   1.50011943 -0.08998492]]

Matriz B:
[[-0.0073093  -0.01390491  0.00368511  0.018839  ]
 [ 0.01769563  0.05777156  0.01447885 -0.08710176]
 [-0.03392627 -0.11353344  0.04679554  0.27048997]]


In [4]:
# Plotando resultados
from lib.plot import colors, plt
from lib.read_data import y_labels

ymin = min(y[:, i].min() for i in range(len(y_labels)))
ymax = max(y[:, i].max() for i in range(len(y_labels)))
for i in range(len(y_labels)):
    plt.scatter(
        cycles,
        y[:, i],
        label=f"{y_labels[i]} (gPROMS)",
        color=colors[i],
        alpha=0.6,
    )
    plt.plot(
        cycles,
        y_arx[:, i],
        label=f"{y_labels[i]} ({model_name})",
        color=colors[i],
    )
plt.ylim(ymin * 0.9, ymax * 1.1)

plt.legend()
plt.ylabel("Valores / %")
plt.xlabel("Ciclo")

plt.savefig(f"../figures/comparision-{model_name}.png")
plt.close()
